# Anomalieerkennung für Payara-Instanzen mit Autoencoder

Dieses Notebook erkennt Anomalien in Betriebsmetriken von **256 Payara-Instanzen**.

**Metriken pro Instanz:**
- CPU-Nutzung (%)
- Speicherbedarf (MB)
- Anzahl aktiver Connections
- HTTP-Response-Zeit (ms)

**Ansatz:** Ein Autoencoder wird auf Normalverhalten trainiert. Anomalien werden über den Rekonstruktionsfehler erkannt — Datenpunkte, die das Netz schlecht rekonstruieren kann, weichen vom gelernten Normalzustand ab.

**Struktur:**
1. Setup und Imports
2. Daten (synthetisch mit eingestreuten Anomalien — für echte Daten einfach den Ladeschritt austauschen)
3. Explorative Analyse
4. Preprocessing
5. Autoencoder-Architektur
6. Training
7. Schwellwertbestimmung
8. Anomalieerkennung und Auswertung
9. Analyse pro Instanz


## 1. Setup und Imports

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.metrics import precision_recall_fscore_support, roc_auc_score, confusion_matrix

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers

# Reproduzierbarkeit
SEED = 42
np.random.seed(SEED)
tf.random.set_seed(SEED)

sns.set_theme(style="whitegrid")
plt.rcParams["figure.figsize"] = (12, 5)

print(f"TensorFlow: {tf.__version__}")
print(f"GPU verfügbar: {bool(tf.config.list_physical_devices('GPU'))}")

## 2. Daten laden bzw. simulieren

In Produktion ersetzt du diesen Block durch das Laden echter Payara-Metriken (z. B. aus Prometheus, InfluxDB, JMX-Export, ELK). Die Zielstruktur ist ein DataFrame mit einer Zeile pro (Instanz, Zeitstempel) und den vier Metrik-Spalten.

Für dieses Notebook simulieren wir realistische Metriken mit Tagesrhythmus und eingestreuten Anomalien (CPU-Spikes, Memory-Leaks, Connection-Floods, Latenz-Ausreißer).

In [ ]:
N_INSTANCES = 256
N_HOURS = 24 * 14              # zwei Wochen Historie
POINTS_PER_HOUR = 12           # 5-Minuten-Auflösung
N_POINTS = N_HOURS * POINTS_PER_HOUR
ANOMALY_RATE = 0.005           # 0,5 % — realistische Rate für Produktionsmonitoring

def simulate_instance(instance_id: int, n_points: int) -> pd.DataFrame:
    """Simuliert Metriken einer Instanz inkl. Tagesrhythmus."""
    t = np.arange(n_points)
    hours = (t / POINTS_PER_HOUR) % 24
    # Tageslast (Sinus): tagsüber mehr, nachts weniger
    daily = 0.5 * (1 + np.sin((hours - 8) / 24 * 2 * np.pi))

    # Instanzspezifische Grundlast, damit Instanzen nicht identisch sind
    rng = np.random.default_rng(SEED + instance_id)
    base_cpu = rng.uniform(15, 35)
    base_mem = rng.uniform(1500, 3500)
    base_conn = rng.uniform(50, 150)
    base_rt  = rng.uniform(80, 180)

    cpu  = base_cpu + 30 * daily + rng.normal(0, 3, n_points)
    mem  = base_mem + 800 * daily + rng.normal(0, 60, n_points)
    conn = base_conn + 200 * daily + rng.normal(0, 12, n_points)
    rt   = base_rt + 40 * daily + rng.normal(0, 10, n_points)

    df = pd.DataFrame({
        "instance_id": instance_id,
        "timestamp": pd.date_range("2025-01-01", periods=n_points, freq="5min"),
        "cpu_percent": np.clip(cpu, 0, 100),
        "memory_mb": np.clip(mem, 0, None),
        "connections": np.clip(conn, 0, None).astype(int),
        "response_time_ms": np.clip(rt, 1, None),
        "is_anomaly": 0,
    })
    return df

def inject_anomalies(df: pd.DataFrame, rate: float) -> pd.DataFrame:
    """Streut vier Anomalietypen ein und markiert sie im Label."""
    df = df.copy()
    rng = np.random.default_rng(SEED)
    n_anom = int(len(df) * rate)
    idx = rng.choice(len(df), size=n_anom, replace=False)

    for i in idx:
        anomaly_type = rng.integers(0, 4)
        if anomaly_type == 0:      # CPU-Spike
            df.loc[i, "cpu_percent"] = min(100, df.loc[i, "cpu_percent"] * rng.uniform(2.5, 4.0))
        elif anomaly_type == 1:    # Memory-Leak
            df.loc[i, "memory_mb"] *= rng.uniform(2.0, 3.5)
        elif anomaly_type == 2:    # Connection-Flood
            df.loc[i, "connections"] = int(df.loc[i, "connections"] * rng.uniform(4.0, 8.0))
        else:                      # Latenz-Ausreißer
            df.loc[i, "response_time_ms"] *= rng.uniform(5.0, 10.0)
        df.loc[i, "is_anomaly"] = 1
    return df

print("Simuliere Daten für 256 Instanzen ...")
frames = [simulate_instance(i, N_POINTS) for i in range(N_INSTANCES)]
data = pd.concat(frames, ignore_index=True)
data = inject_anomalies(data, ANOMALY_RATE)

print(f"Gesamt: {len(data):,} Messpunkte | Anomalien: {data['is_anomaly'].sum():,} ({data['is_anomaly'].mean()*100:.2f} %)")
data.head()

## 3. Explorative Analyse

In [ ]:
FEATURES = ["cpu_percent", "memory_mb", "connections", "response_time_ms"]

print("Deskriptive Statistik:")
data[FEATURES].describe().round(2)

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 8))
for ax, feat in zip(axes.ravel(), FEATURES):
    ax.hist(data.loc[data.is_anomaly == 0, feat], bins=60, alpha=0.6, label="normal", density=True)
    ax.hist(data.loc[data.is_anomaly == 1, feat], bins=60, alpha=0.6, label="anomal",  density=True)
    ax.set_title(feat); ax.legend()
plt.suptitle("Verteilung pro Metrik: normal vs. anomal", y=1.02)
plt.tight_layout(); plt.show()

In [ ]:
# Beispiel: Zeitverlauf einer Instanz
sample = data[data.instance_id == 0].head(1000)
fig, axes = plt.subplots(4, 1, figsize=(14, 9), sharex=True)
for ax, feat in zip(axes, FEATURES):
    ax.plot(sample["timestamp"], sample[feat], linewidth=0.8)
    anom = sample[sample.is_anomaly == 1]
    ax.scatter(anom["timestamp"], anom[feat], color="red", s=25, zorder=5, label="Anomalie")
    ax.set_ylabel(feat); ax.legend(loc="upper right")
axes[0].set_title("Instanz 0 — Metriken über die Zeit")
plt.tight_layout(); plt.show()

## 4. Preprocessing

- Feature-Matrix `X` extrahieren
- Train/Validation/Test-Split
- **Wichtig:** Das Training läuft nur auf den als *normal* markierten Punkten — der Autoencoder soll ausschließlich das Normalverhalten lernen.
- Standardisierung mit `StandardScaler` (Fit nur auf Trainings-Normaldaten, um Data Leakage zu vermeiden).

In [ ]:
X = data[FEATURES].values
y = data["is_anomaly"].values

# Train / Test-Split (stratifiziert)
X_train_full, X_test, y_train_full, y_test = train_test_split(
    X, y, test_size=0.2, random_state=SEED, stratify=y
)

# Nur normale Daten fürs Training verwenden
X_train_normal = X_train_full[y_train_full == 0]

# Skalierung ausschließlich auf Trainings-Normaldaten fitten
scaler = StandardScaler().fit(X_train_normal)
X_train_scaled = scaler.transform(X_train_normal)
X_test_scaled  = scaler.transform(X_test)

# Interne Validierung während des Trainings
X_tr, X_val = train_test_split(X_train_scaled, test_size=0.15, random_state=SEED)

print(f"Training  (nur normal): {X_tr.shape}")
print(f"Validation (nur normal): {X_val.shape}")
print(f"Test (gemischt)       : {X_test_scaled.shape}  |  Anomalien im Test: {y_test.sum()}")

## 5. Autoencoder-Architektur

Symmetrischer Encoder-Decoder mit Bottleneck. Bei 4 Eingabefeatures reicht ein kleines Netz:

`4 → 8 → 4 → 2 (bottleneck) → 4 → 8 → 4`

Wenige Neuronen im Bottleneck erzwingen, dass nur die wesentliche Struktur der Normaldaten komprimiert wird. Anomale Punkte, die nicht diesem Muster folgen, lassen sich schlecht rekonstruieren — daraus wird der Anomalie-Score.

In [ ]:
def build_autoencoder(input_dim: int, bottleneck: int = 2) -> keras.Model:
    inputs = keras.Input(shape=(input_dim,), name="input")
    x = layers.Dense(8, activation="relu")(inputs)
    x = layers.Dense(4, activation="relu")(x)
    z = layers.Dense(bottleneck, activation="relu", name="bottleneck")(x)
    x = layers.Dense(4, activation="relu")(z)
    x = layers.Dense(8, activation="relu")(x)
    outputs = layers.Dense(input_dim, activation="linear", name="reconstruction")(x)

    model = keras.Model(inputs, outputs, name="autoencoder")
    model.compile(optimizer=keras.optimizers.Adam(1e-3), loss="mse", metrics=["mae"])
    return model

autoencoder = build_autoencoder(input_dim=len(FEATURES))
autoencoder.summary()

## 6. Training

In [ ]:
callbacks = [
    keras.callbacks.EarlyStopping(patience=10, restore_best_weights=True, monitor="val_loss"),
    keras.callbacks.ReduceLROnPlateau(patience=5, factor=0.5, min_lr=1e-5, monitor="val_loss"),
]

history = autoencoder.fit(
    X_tr, X_tr,               # Autoencoder: Input == Target
    validation_data=(X_val, X_val),
    epochs=100,
    batch_size=256,
    callbacks=callbacks,
    verbose=1,
)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 4))
axes[0].plot(history.history["loss"], label="train")
axes[0].plot(history.history["val_loss"], label="validation")
axes[0].set_title("Loss (MSE)"); axes[0].set_xlabel("Epoch"); axes[0].legend()

axes[1].plot(history.history["mae"], label="train")
axes[1].plot(history.history["val_mae"], label="validation")
axes[1].set_title("MAE"); axes[1].set_xlabel("Epoch"); axes[1].legend()
plt.tight_layout(); plt.show()

## 7. Rekonstruktionsfehler und Schwellwert

Der Rekonstruktionsfehler pro Punkt ist der MSE zwischen Input und Rekonstruktion. Punkte mit hohem Fehler gelten als anomal.

**Schwellwert:** Wir berechnen den Fehler auf reinen Normaldaten und wählen ein hohes Perzentil (z. B. 99 %) als Grenze. So wird die False-Positive-Rate auf ~1 % begrenzt, unabhängig vom Testset.

In [ ]:
def reconstruction_error(model, X):
    X_pred = model.predict(X, verbose=0)
    return np.mean(np.square(X - X_pred), axis=1)

# Fehler auf Normaldaten (Validierung) für den Schwellwert
# 99,9 % Perzentil → nur ~0,1 % False-Positive-Rate. Echte Spikes liegen bei
# einem gut trainierten Autoencoder um Größenordnungen über dem Schwellwert
# und werden davon nicht beeinträchtigt — nur die Grenzfälle im Rauschen fallen weg.
THRESHOLD_PERCENTILE = 99.9
val_errors = reconstruction_error(autoencoder, X_val)
threshold  = np.percentile(val_errors, THRESHOLD_PERCENTILE)
print(f"Schwellwert ({THRESHOLD_PERCENTILE}. Perzentil der Normaldaten): {threshold:.6f}")

# Fehler auf Testset
test_errors = reconstruction_error(autoencoder, X_test_scaled)

fig, ax = plt.subplots(figsize=(12, 5))
ax.hist(test_errors[y_test == 0], bins=80, alpha=0.6, label="normal",  density=True)
ax.hist(test_errors[y_test == 1], bins=80, alpha=0.6, label="anomal",  density=True)
ax.axvline(threshold, color="red", linestyle="--", label=f"Schwellwert = {threshold:.4f}")
ax.set_xlabel("Rekonstruktionsfehler (MSE)"); ax.set_ylabel("Dichte")
ax.set_title("Verteilung der Rekonstruktionsfehler auf dem Testset")
ax.set_xscale("log"); ax.legend()
plt.tight_layout(); plt.show()

## 8. Anomalieerkennung und Auswertung

In [ ]:
y_pred = (test_errors > threshold).astype(int)

precision, recall, f1, _ = precision_recall_fscore_support(y_test, y_pred, average="binary")
auc = roc_auc_score(y_test, test_errors)  # AUC auf kontinuierlichem Score
cm  = confusion_matrix(y_test, y_pred)

print(f"Precision : {precision:.3f}")
print(f"Recall    : {recall:.3f}")
print(f"F1-Score  : {f1:.3f}")
print(f"ROC-AUC   : {auc:.3f}")

fig, ax = plt.subplots(figsize=(5, 4))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues",
            xticklabels=["normal", "anomal"], yticklabels=["normal", "anomal"], ax=ax)
ax.set_xlabel("Vorhersage"); ax.set_ylabel("Tatsächlich"); ax.set_title("Konfusionsmatrix")
plt.tight_layout(); plt.show()

In [ ]:
# Precision-Recall-Kurve über verschiedene Schwellwerte
from sklearn.metrics import precision_recall_curve, auc as sk_auc

prec_curve, rec_curve, thresh_curve = precision_recall_curve(y_test, test_errors)
pr_auc = sk_auc(rec_curve, prec_curve)

fig, ax = plt.subplots(figsize=(7, 5))
ax.plot(rec_curve, prec_curve, linewidth=2)
ax.set_xlabel("Recall"); ax.set_ylabel("Precision")
ax.set_title(f"Precision-Recall-Kurve (AUC = {pr_auc:.3f})")
plt.tight_layout(); plt.show()

## 9. Analyse pro Instanz

In der Praxis interessiert nicht nur, *ob* etwas anomal ist, sondern *wo*. Wir aggregieren die Rekonstruktionsfehler auf allen Daten pro Instanz — so lassen sich Problemkinder identifizieren.

In [ ]:
# Fehler auf dem gesamten Datensatz berechnen
X_all_scaled = scaler.transform(X)
all_errors = reconstruction_error(autoencoder, X_all_scaled)

result = data.copy()
result["reconstruction_error"] = all_errors
result["predicted_anomaly"] = (all_errors > threshold).astype(int)

instance_stats = result.groupby("instance_id").agg(
    mean_error=("reconstruction_error", "mean"),
    max_error=("reconstruction_error", "max"),
    anomaly_count=("predicted_anomaly", "sum"),
    total_points=("predicted_anomaly", "size"),
).reset_index()
instance_stats["anomaly_rate_pct"] = 100 * instance_stats["anomaly_count"] / instance_stats["total_points"]

print("Top-10-Instanzen mit den meisten erkannten Anomalien:")
instance_stats.sort_values("anomaly_count", ascending=False).head(10)

### Persistenz-Filter gegen Einzelausreißer

Einzelne Ausreißer sind für den Betrieb meist irrelevant (Messrauschen, kurzer GC-Peak). Wir zählen nur solche Ereignisse als echte **Incidents**, bei denen die Anomalie mindestens `MIN_PERSISTENCE` Messpunkte in Folge auftritt (bei 5-Minuten-Auflösung → 25 min Dauer).

In [ ]:
MIN_PERSISTENCE = 5  # min. Punkte in Folge (bei 5-min-Auflösung = 25 min)

def find_incidents(df: pd.DataFrame, min_len: int) -> pd.DataFrame:
    """Findet zusammenhängende Anomalie-Läufe pro Instanz."""
    df = df.sort_values(["instance_id", "timestamp"]).reset_index(drop=True)
    incidents = []
    for inst_id, g in df.groupby("instance_id"):
        flags = g["predicted_anomaly"].values
        i = 0
        while i < len(flags):
            if flags[i] == 1:
                j = i
                while j < len(flags) and flags[j] == 1:
                    j += 1
                run_len = j - i
                if run_len >= min_len:
                    incidents.append({
                        "instance_id": inst_id,
                        "start":       g.iloc[i]["timestamp"],
                        "end":         g.iloc[j-1]["timestamp"],
                        "duration_min": run_len * 5,
                        "max_error":   float(g.iloc[i:j]["reconstruction_error"].max()),
                    })
                i = j
            else:
                i += 1
    return pd.DataFrame(incidents)

incidents = find_incidents(result, MIN_PERSISTENCE)
print(f"Rohe Anomalie-Punkte: {result['predicted_anomaly'].sum():,}")
print(f"Echte Incidents nach Persistenz-Filter ({MIN_PERSISTENCE}+ Punkte in Folge): {len(incidents):,}")
print(f"Betroffene Instanzen: {incidents['instance_id'].nunique() if len(incidents) else 0} / {N_INSTANCES}")

if len(incidents):
    print("\nTop-10 längste Incidents:")
    print(incidents.sort_values("duration_min", ascending=False).head(10).to_string(index=False))

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(15, 5))

top20 = instance_stats.sort_values("anomaly_count", ascending=False).head(20)
axes[0].barh(top20["instance_id"].astype(str), top20["anomaly_count"], color="steelblue")
axes[0].set_xlabel("Anzahl erkannter Anomalien")
axes[0].set_ylabel("Instanz-ID")
axes[0].set_title("Top-20 Instanzen nach Anomaliezahl")
axes[0].invert_yaxis()

axes[1].hist(instance_stats["anomaly_rate_pct"], bins=30, color="coral", edgecolor="black")
axes[1].set_xlabel("Anomalierate (%)"); axes[1].set_ylabel("Anzahl Instanzen")
axes[1].set_title("Verteilung der Anomalierate über 256 Instanzen")

plt.tight_layout(); plt.show()

In [ ]:
# Detailansicht der auffälligsten Instanz
worst_id = instance_stats.sort_values("anomaly_count", ascending=False).iloc[0]["instance_id"]
worst = result[result.instance_id == worst_id].head(1500)

fig, axes = plt.subplots(4, 1, figsize=(14, 10), sharex=True)
for ax, feat in zip(axes, FEATURES):
    ax.plot(worst["timestamp"], worst[feat], linewidth=0.8, color="steelblue")
    detected = worst[worst.predicted_anomaly == 1]
    ax.scatter(detected["timestamp"], detected[feat], color="red", s=20, zorder=5, label="erkannte Anomalie")
    ax.set_ylabel(feat); ax.legend(loc="upper right")
axes[0].set_title(f"Auffälligste Instanz (ID = {int(worst_id)})")
plt.tight_layout(); plt.show()

## 10. Modell speichern und in Produktion nutzen

In [ ]:
# Modell + Scaler speichern
autoencoder.save("payara_autoencoder.keras")

import joblib
joblib.dump(scaler, "payara_scaler.pkl")
joblib.dump({"threshold": float(threshold), "features": FEATURES}, "payara_config.pkl")

print("Modell, Scaler und Konfiguration gespeichert.")

In [ ]:
def detect_anomaly(cpu, memory, connections, response_time,
                   model=autoencoder, scaler=scaler, threshold=threshold):
    """Anomaliecheck für einen einzelnen Metrik-Snapshot."""
    x = np.array([[cpu, memory, connections, response_time]])
    x_scaled = scaler.transform(x)
    x_pred = model.predict(x_scaled, verbose=0)
    error = float(np.mean((x_scaled - x_pred) ** 2))
    return {
        "reconstruction_error": error,
        "threshold": float(threshold),
        "is_anomaly": bool(error > threshold),
        "severity": float(error / threshold),  # >1 = anomal, größer = extremer
    }

# Beispiele
print("Normalbetrieb:", detect_anomaly(cpu=35, memory=2500, connections=180, response_time=140))
print("CPU-Spike   :", detect_anomaly(cpu=98, memory=2500, connections=180, response_time=140))
print("Latenz-Peak :", detect_anomaly(cpu=35, memory=2500, connections=180, response_time=2000))

## Nächste Schritte für den Produktivbetrieb

- **Datenquelle anbinden:** Simulation ersetzen durch Live-Feed (Prometheus/Micrometer-Scrape, JMX, Payara-REST-Admin, Log-Pipeline).
- **Sliding-Window-Features:** Statt Einzelpunkten Fenster (5, 15, 60 min) als Input verwenden — dann werden auch schleichende Trends erkannt (z. B. langsame Memory-Leaks).
- **Instanz-Embedding:** Instanz-ID als kategoriales Feature (Embedding-Layer) mitgeben, wenn Instanzen unterschiedliche Rollen haben (Frontend vs. Batch-Worker).
- **Regelmäßiges Retraining:** Wöchentlich/monatlich, um Drift der Normalverteilung nachzuziehen.
- **Schwellwert-Tuning:** 99 % ist ein Ausgangspunkt. Recall vs. False-Positive-Rate am Business-Impact ausrichten (Alarmmüdigkeit vs. verpasste Incidents).
- **Alerting:** Anomalie-Events an Prometheus Alertmanager, PagerDuty oder Slack pushen — inklusive Severity und der Instanz-ID, damit direkt eskaliert werden kann.
- **Alternative Modelle vergleichen:** Isolation Forest, LSTM-Autoencoder (für stärker sequenzielle Muster) oder Variational Autoencoder als Baseline.
